# IMDb Movie Rating Prediction

## Machine Learning Project

### Problem Statement
The objective of this project is to predict the IMDb rating of a movie using
available movie information such as genre, director, year, runtime and number
of votes.

Since IMDb Rating is a continuous numerical value, the main problem is treated
as a **Regression problem**.

In addition, a classification analysis is performed by dividing movies into
two categories:
- Good Movie: Rating >= 7
- Not Good Movie: Rating < 7

This classification part is used to understand TP, TN, FP, FN and confusion
matrices.

## Importing Required Libraries

I import the libraries required for data analysis, visualization and machine
learning.

Pandas is used for handling the dataset, NumPy for numerical operations,
Matplotlib and Seaborn for visualization, and Scikit-learn for building and
evaluating machine learning models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

## Loading the Dataset

I load the IMDb movie dataset using Pandas.

After loading the dataset, I display its shape and first few rows to make sure
the file has been loaded correctly and to understand its basic structure.

In [ ]:
df = pd.read_csv("IMDB-Movie-Data-selected-columns.csv")

print("Dataset Shape:", df.shape)
display(df.head())

## Understanding the Dataset

Before applying machine learning, I first understand the columns and their
data types.

This helps me identify which columns are numerical and which are categorical.
It also helps me decide which features can be used for prediction.

In [ ]:
print("Column Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nDataset Information:")
df.info()

## Checking Data Quality

I check for missing values and duplicate rows.

Missing values can cause problems during model training, while duplicate
records may give unnecessary importance to the same movie.

The dataset does not contain missing values or duplicate rows, so no special
treatment is required.

In [ ]:
print("Missing Values:")
display(df.isnull().sum())

print("Number of Duplicate Rows:", df.duplicated().sum())

## Exploratory Data Analysis

I perform some basic exploratory analysis to understand the distribution of
IMDb ratings.

The histogram shows how movie ratings are distributed in the dataset.

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(df["Rating"], bins=20, kde=True)

plt.title("Distribution of IMDb Ratings")
plt.xlabel("IMDb Rating")
plt.ylabel("Number of Movies")

plt.show()

print("Average Rating:", round(df["Rating"].mean(), 2))
print("Minimum Rating:", df["Rating"].min())
print("Maximum Rating:", df["Rating"].max())

## Feature Selection

The target variable is Rating.

I selected Genre, Director, Year, Runtime and Votes as input features.

I did not use Title, Description and Actors because these are text-heavy
columns and would require Natural Language Processing techniques.

Rank is also excluded because it represents the movie's position in the
dataset rather than a direct movie characteristic.

For Genre, I use the first listed genre to keep the feature simple and
easy to interpret.

In [ ]:
data = df.copy()

data["Main_Genre"] = (
    data["Genre"]
    .str.split(",")
    .str[0]
    .str.strip()
)

features = [
    "Main_Genre",
    "Director",
    "Year",
    "Runtime (Minutes)",
    "Votes"
]

X = data[features]
y = data["Rating"]

display(X.head())

## Splitting the Dataset

I divide the data into training and testing sets.

80% of the data is used for training and 20% is kept for testing.

The test data is not used during training, so it gives us an estimate of how
the model performs on unseen movies.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

## Data Preprocessing

Machine learning algorithms cannot directly work with categorical text
values such as genre and director.

Therefore, I use One-Hot Encoding to convert categorical features into
numerical form.

Numerical features such as Year, Runtime and Votes are passed directly to
the models.

In [ ]:
categorical_features = [
    "Main_Genre",
    "Director"
]

numerical_features = [
    "Year",
    "Runtime (Minutes)",
    "Votes"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "numerical",
            "passthrough",
            numerical_features
        )
    ]
)

print("Preprocessing setup completed.")

## Building Regression Models

Since IMDb Rating is a continuous numerical value, I use regression
algorithms.

I compare three models:
1. Linear Regression
2. Random Forest Regressor
3. Gradient Boosting Regressor

Using multiple models allows me to compare their performance and select the
best one based on evaluation metrics.

In [ ]:
regression_models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        random_state=42
    )
}

trained_regression_models = {}

for name, model in regression_models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    trained_regression_models[name] = pipeline

print("All regression models trained successfully.")

## Evaluating Regression Models

I evaluate the models using MAE, RMSE and R² score.

MAE represents the average prediction error.

RMSE gives more importance to larger errors.

R² indicates how well the model explains the variation in IMDb ratings.

For MAE and RMSE, lower values are better. For R², a higher value is better.

In [ ]:
regression_results = []

for name, model in trained_regression_models.items():

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    regression_results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })

regression_results_df = pd.DataFrame(regression_results)

display(regression_results_df.round(3))

## Selecting the Best Regression Model

I select the model with the lowest RMSE because RMSE directly measures the
prediction error and gives additional importance to larger errors.

The best model is selected from the actual results instead of assuming that
one algorithm will always perform best.

In [ ]:
best_regression_model_name = (
    regression_results_df
    .sort_values("RMSE")
    .iloc[0]["Model"]
)

best_regression_model = trained_regression_models[
    best_regression_model_name
]

print("Best Regression Model:", best_regression_model_name)

## Actual vs Predicted Ratings

To visually understand the performance of the best regression model, I
compare the actual ratings with the predicted ratings.

Points closer to the diagonal line indicate predictions that are closer to
the actual IMDb ratings.

In [ ]:
best_predictions = best_regression_model.predict(X_test)

plt.figure(figsize=(7, 5))
plt.scatter(y_test, best_predictions, alpha=0.6)

plt.xlabel("Actual Rating")
plt.ylabel("Predicted Rating")
plt.title(f"Actual vs Predicted - {best_regression_model_name}")

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle="--"
)

plt.show()

# Additional Classification Analysis

## Converting Rating into Classes

The main project is a regression problem.

However, confusion matrices and TP, TN, FP and FN are classification
concepts.

Therefore, for additional analysis, I convert the numerical IMDb rating
into two classes:

- Rating >= 7.0 → Good Movie (1)
- Rating < 7.0 → Not Good Movie (0)

This allows me to perform classification and calculate TP, TN, FP and FN.

In [ ]:
data["Rating_Class"] = np.where(
    data["Rating"] >= 7.0,
    1,
    0
)

X_class = data[features]
y_class = data["Rating_Class"]

print("0 = Not Good Movie")
print("1 = Good Movie")

print("\nClass Distribution:")
print(y_class.value_counts())

## Classification Train/Test Split

I split the classification data into training and testing sets.

I use stratification so that the proportion of Good and Not Good movies is
maintained in both datasets.

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class,
    y_class,
    test_size=0.20,
    random_state=42,
    stratify=y_class
)

print("Classification Training Samples:", len(X_train_c))
print("Classification Testing Samples:", len(X_test_c))

## Building Classification Models

For classification, I use three algorithms:
1. Logistic Regression
2. Random Forest Classifier
3. Gradient Boosting Classifier

These models predict whether a movie belongs to the Good Movie or Not Good
Movie category.

In [ ]:
classification_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        random_state=42
    )
}

trained_classification_models = {}

for name, model in classification_models.items():

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train_c, y_train_c)

    trained_classification_models[name] = pipeline

print("All classification models trained successfully.")

## Confusion Matrix

A confusion matrix shows how the model's predictions compare with the actual
classes.

For this project:
- TN: Actual Not Good and predicted Not Good
- FP: Actual Not Good but predicted Good
- FN: Actual Good but predicted Not Good
- TP: Actual Good and predicted Good

A good classification model should have high TP and TN and low FP and FN.

In [ ]:
for name, model in trained_classification_models.items():

    y_pred = model.predict(X_test_c)

    cm = confusion_matrix(y_test_c, y_pred)

    TN, FP, FN, TP = cm.ravel()

    print("=" * 50)
    print(name)
    print("=" * 50)

    print("True Negative (TN):", TN)
    print("False Positive (FP):", FP)
    print("False Negative (FN):", FN)
    print("True Positive (TP):", TP)

## Visualizing Confusion Matrices

I use a confusion matrix to visually understand the classification results
of each model.

Rows represent the actual class and columns represent the predicted class.

Class 0 represents Not Good Movie and Class 1 represents Good Movie.

In [ ]:
for name, model in trained_classification_models.items():

    y_pred = model.predict(X_test_c)

    ConfusionMatrixDisplay.from_predictions(
        y_test_c,
        y_pred,
        display_labels=["Not Good", "Good"],
        values_format="d"
    )

    plt.title(f"Confusion Matrix - {name}")
    plt.show()

## Classification Model Evaluation

I compare the models using Accuracy, Precision, Recall and F1 Score.

Accuracy measures the overall percentage of correct predictions.

Precision tells us how many movies predicted as Good were actually Good.

Recall tells us how many actual Good movies were correctly identified.

F1 Score combines precision and recall into one measure.

In [ ]:
classification_results = []

for name, model in trained_classification_models.items():

    y_pred = model.predict(X_test_c)

    classification_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_c, y_pred),
        "Precision": precision_score(y_test_c, y_pred),
        "Recall": recall_score(y_test_c, y_pred),
        "F1 Score": f1_score(y_test_c, y_pred)
    })

classification_results_df = pd.DataFrame(
    classification_results
)

display(classification_results_df.round(3))

## TP, TN, FP and FN Comparison

This table compares the four types of predictions made by all classification
models.

True Positives and True Negatives represent correct predictions.

False Positives and False Negatives represent incorrect predictions.

This comparison helps me understand not only how accurate the models are,
but also what types of mistakes they make.

In [ ]:
confusion_results = []

for name, model in trained_classification_models.items():

    y_pred = model.predict(X_test_c)

    TN, FP, FN, TP = confusion_matrix(
        y_test_c,
        y_pred
    ).ravel()

    confusion_results.append({
        "Model": name,
        "TN": TN,
        "FP": FP,
        "FN": FN,
        "TP": TP
    })

confusion_results_df = pd.DataFrame(
    confusion_results
)

display(confusion_results_df)

## Selecting the Best Classification Model

I select the model with the highest F1 Score.

F1 Score is useful because it considers both precision and recall.

The model is selected based on the actual test results rather than assuming
that a particular algorithm will perform best.

In [ ]:
best_classification_model_name = (
    classification_results_df
    .sort_values("F1 Score", ascending=False)
    .iloc[0]["Model"]
)

print(
    "Best Classification Model:",
    best_classification_model_name
)

## Sample Prediction

Finally, I test the trained regression model using a new movie record.

The movie was not part of the test dataset.

The model uses the movie's genre, director, year, runtime and number of votes
to estimate its IMDb rating.

In [ ]:
new_movie = pd.DataFrame({
    "Main_Genre": ["Action"],
    "Director": ["Christopher Nolan"],
    "Year": [2026],
    "Runtime (Minutes)": [140],
    "Votes": [250000]
})

predicted_rating = best_regression_model.predict(
    new_movie
)[0]

print(
    "Predicted IMDb Rating:",
    round(predicted_rating, 2)
)

# Final Conclusion

In this project, I developed an end-to-end machine learning system using the
IMDb movie dataset.

I first understood the dataset and performed data quality checks. The dataset
contains 1000 movies and there are no missing values or duplicate records.

For the main problem, I treated IMDb Rating as a continuous numerical target,
so I used regression algorithms.

I trained Linear Regression, Random Forest Regressor and Gradient Boosting
Regressor and compared them using MAE, RMSE and R² Score.

The model with the lowest RMSE was selected as the best regression model.

As an additional analysis, I converted movie ratings into two categories:
Good Movie and Not Good Movie. This allowed me to use classification models
and evaluate them using a confusion matrix, TP, TN, FP, FN, Accuracy,
Precision, Recall and F1 Score.

Finally, I used the selected regression model to predict the IMDb rating of
a new unseen movie.

Therefore, the project covers the complete machine learning workflow from
data understanding and preprocessing to model training, evaluation,
comparison and prediction.